# ga-pp-train

**Select GPU T4 ×2, enable Internet, attach `ga-pp-packed`, then run all cells in order.**
The five numbered sections write a reusable Python repository into `/kaggle/working/code/`,
prepare local caches once, run unit checks, train A/B in parallel, and collect results.
Change `VARIANTS` and `SMOKE` in cell 3. A–D are implemented; longer lists run in two-GPU waves.

Defaults: 224 RGB input, batch 32, AdamW lr 1e-3 / weight decay 1e-4, six epochs,
500-step warmup, AMP fp16, seed 0, two loader workers and four hours per variant.
Smoke mode still builds full caches, but trains 300 steps and evaluates 200 validation/test samples.

All large copies, decoded images, text vectors, optional patch tokens, CLIP weights and transient
run outputs live under `/tmp`. C/D token construction needs at least 25 GB free on `/tmp`.
Only code and collected checkpoints/logs/metrics/figures go into `/kaggle/working` (under 15 GB).
The notebook copies the dataset's metric and packed-image helper **verbatim** into the repo.
No new success metric is introduced, and CLIP is frozen and used only during preparation.

Before training, the exact target maps must pass the fixed 200-validation-example round-trip
check at ≥95% using the supplied metric. A failed check saves diagnostics and stops execution.
During training, every epoch uses the same seed-0 validation subset for checkpoint selection.
Final validation/test results use the best checkpoint. Wall-time stopping finishes its pending
validation and then performs the final evaluations. Progress is flushed plain text; launcher
snapshots print the last log lines every 60 seconds for Kaggle commit logs.

Kaggle must execute the GPU runs; this notebook contains no precomputed training scores.
Sources for the cache APIs: [HF CLIP](https://huggingface.co/docs/transformers/model_doc/clip),
[ViT-B/16 config](https://huggingface.co/openai/clip-vit-base-patch16/blob/main/config.json),
[torchvision blur](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.functional.gaussian_blur.html).


## Cell 1 — Setup and GPU/data discovery


In [ ]:
# CELL 1 — T4 x2 setup; attach ga-pp-packed; all large caches use /tmp.
import os, sys, glob, shutil, subprocess, importlib.util, hashlib
from pathlib import Path
os.environ['HF_HOME']='/tmp/ga_hf'
os.environ['TORCH_HOME']='/tmp/ga_torch'
os.environ['MPLCONFIGDIR']='/tmp/ga_mpl'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS']='1'
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['PYTHONUNBUFFERED']='1'
if importlib.util.find_spec('cv2') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir','--no-deps','opencv-python-headless'],check=True)
required=['torch','torchvision','transformers','numpy','pandas','pyarrow','PIL','shapely','matplotlib']
missing=[name for name in required if importlib.util.find_spec(name) is None]
if missing:raise RuntimeError(f'Use the standard Kaggle PyTorch GPU image; missing libraries: {missing}')
import torch, cv2
subprocess.run(['nvidia-smi'],check=True)
if torch.cuda.device_count()<2:raise RuntimeError('Select Kaggle Accelerator -> GPU T4 x2 (two visible CUDA devices).')
print('GPUs:',[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],flush=True)
matches=sorted(glob.glob('/kaggle/input/**/ga_eval/packed/images.bin',recursive=True))
if not matches:raise FileNotFoundError('Attach ga-pp-packed: no ga_eval/packed/images.bin found under /kaggle/input.')
ROOT=Path(matches[0]).parent.parent
CODE=Path('/kaggle/working/code');CODE.mkdir(parents=True,exist_ok=True)
for filename in ('eval_utils.py','packed_io.py'):
    source=ROOT/filename
    if not source.is_file():raise FileNotFoundError(source)
    shutil.copyfile(source,CODE/filename)
print('ROOT:',ROOT,'| repository:',CODE,flush=True)
print('Copied dataset eval_utils.py and packed_io.py verbatim; no metric reimplementation.',flush=True)
optional_token=None
try:
    from kaggle_secrets import UserSecretsClient
    optional_token=UserSecretsClient().get_secret('HF_TOKEN')
except Exception as error:
    print('Optional HF secret unavailable:',type(error).__name__,flush=True)
if optional_token:
    os.environ['HF_TOKEN']=optional_token
    print('Optional HF authentication configured (credential hidden).',flush=True)
else:
    os.environ.pop('HF_TOKEN',None)
    print('Hugging Face access is anonymous.',flush=True)
del optional_token
print('Setup complete. Large data/model caches stay in /tmp; outputs are capped below 15 GB.',flush=True)


## Cell 2.1 — Write `code/config.py`


In [ ]:
%%writefile /kaggle/working/code/config.py
"""Experiment defaults. Large caches and transient training runs stay outside working/."""
from dataclasses import dataclass, field, asdict

@dataclass
class Config:
    variants: list[str] = field(default_factory=lambda: ['A', 'B'])
    variant: str = 'A'
    img: int = 224
    batch: int = 32
    lr: float = 1e-3
    wd: float = 1e-4
    epochs: int = 6
    max_hours: float = 3.5
    warmup_steps: int = 500
    amp: bool = True
    seed: int = 0
    num_workers: int = 2
    width_norm: float = 150.0
    min_rect_px: float = 2.0
    lambda_gate: float = 0.5
    val_subset: int = 1000
    smoke: bool = False
    pack_root: str = '/tmp/ga_pack'
    cache: str = '/tmp/ga_cache'
    out: str = '/tmp/ga_runs/A'
    log_every: int = 200

    def validate(self):
        if self.variant not in 'ABCDE' or len(self.variant) != 1:
            raise ValueError('variant must be A, B, C, D or E (E = C0: text-free CLIP gate)')
        if self.img != 224:
            raise ValueError('This architecture/cache contract requires img=224')
        if self.batch < 1 or self.epochs < 1 or self.max_hours <= 0:
            raise ValueError('batch, epochs and max_hours must be positive')
        return self

    def to_dict(self):
        return asdict(self)


## Cell 2.2 — Write `code/common.py`


In [ ]:
%%writefile /kaggle/working/code/common.py
import hashlib
import json
import os
import random
import shutil
from pathlib import Path
import numpy as np
import torch

def log(message):
    print(message, flush=True)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def save_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    pending = path.with_name(path.name+'.partial')
    pending.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    pending.replace(path)

def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(4*1024**2), b''):
            digest.update(block)
    return digest.hexdigest()

def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def worker_seed(worker_id):
    import cv2
    cv2.setNumThreads(0)
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed); random.seed(seed)

def require_free(path, needed, context):
    free = shutil.disk_usage(path).free
    if free < needed:
        raise RuntimeError(f'{context}: need {needed/1e9:.2f} GB free at {path}; available {free/1e9:.2f} GB')

def working_budget(additional=0):
    root = Path('/kaggle/working')
    if root.exists():
        used = sum(path.stat().st_size for path in root.rglob('*') if path.is_file())
        if used+additional >= 15_000_000_000:
            raise RuntimeError('The /kaggle/working 15 GB output budget would be exceeded')

def run_variant(spec):
    """Run specs: 'A'..'E' (seed 0) or e.g. 'C_s1' (variant C, seed 1). E is C0, the text-free CLIP gate control."""
    import re
    match = re.fullmatch(r'([ABCDE])(?:_s(\d+))?', str(spec))
    if match is None: raise ValueError(f'Invalid run spec {spec!r}; use A-E or e.g. C_s1')
    return match.group(1)

def run_seed(spec):
    run_variant(spec)
    return int(spec.split('_s')[1]) if '_s' in spec else 0

def text_batch(cache, indices, device):
    rows = indices.detach().cpu().numpy() if isinstance(indices, torch.Tensor) else np.asarray(indices)
    return torch.from_numpy(np.asarray(cache[rows], dtype=np.float32)).to(device, non_blocking=True)

def model_inputs(batch, text_cache, variant, device):
    image = batch['img'].to(device, non_blocking=True)
    text = None if variant == 'A' else text_batch(text_cache, batch['prompt_idx'], device)
    tokens = batch.get('clip_tokens')
    tokens = tokens.to(device, non_blocking=True) if tokens is not None else None
    return image, text, tokens


## Cell 2.3 — Write `code/prepare.py`


In [ ]:
%%writefile /kaggle/working/code/prepare.py
"""One-time LOCAL copy, CRC-checked image decode, leakage fix and frozen CLIP caches."""
import argparse
import gc
import io
import multiprocessing as mp
import os
import shutil
import string
import time
import unicodedata
import zlib
from pathlib import Path
from common import run_variant
import numpy as np
import pandas as pd
import torch
from PIL import Image
from common import log, read_json, save_json, sha_file, fingerprint, require_free

CLIP_ID = 'openai/clip-vit-base-patch16'
VERBS = sorted(['take hold of','get hold of','grab hold of','pick up','grasp','grab','grip','hold','take',
    'lift','fetch','seize','catch','clutch','give me','bring me','hand me','pass me','get'],key=len,reverse=True)
SEPARATORS = [' by its ',' at its ',' on its ',' by the ',' at the ',' on the ',' from its ',' with its ']

def normalize_name(prompt):
    text = ''.join(' ' if char in string.punctuation or unicodedata.category(char).startswith('P') else char
                   for char in str(prompt).lower())
    text = ' '.join(text.split())
    for verb in VERBS:
        if text == verb or text.startswith(verb+' '): text = text[len(verb):].lstrip(); break
    while text.split(' ',1)[0] in ('the','a','an'): text = text.split(' ',1)[1] if ' ' in text else ''
    positions = [position for sep in SEPARATORS if (position := text.find(sep)) >= 0]
    name = (text[:min(positions)] if positions else text).strip()
    if len(name)>3 and name.endswith('s') and not name.endswith('ss'): name=name[:-1]
    if not name: raise ValueError(f'Empty normalized object name for prompt {prompt!r}')
    return name

def copy_pack(root, destination):
    destination = Path(destination); (destination/'packed').mkdir(parents=True,exist_ok=True)
    old = read_json(destination/'copy.json') if (destination/'copy.json').is_file() else {}
    inventory = {}
    for source in sorted((root/'packed').iterdir()):
        if not source.is_file(): continue
        stat = source.stat()
        identity = [str(root), stat.st_size, stat.st_mtime_ns]
        target = destination/'packed'/source.name
        if target.is_file() and target.stat().st_size == stat.st_size and old.get(source.name)==identity:
            log(f'copy {source.name}: cached ({stat.st_size/1e6:.1f} MB)')
        else:
            require_free(destination,stat.st_size+200_000_000,'Copy packed source')
            started=time.perf_counter(); pending=target.with_name(target.name+'.partial')
            shutil.copyfile(source,pending)
            if pending.stat().st_size != stat.st_size: raise IOError(f'Incomplete copy: {source.name}')
            pending.replace(target)
            elapsed=time.perf_counter()-started
            log(f'copy {source.name}: {stat.st_size/1e6:.1f} MB in {elapsed:.2f}s ({stat.st_size/max(elapsed,1e-6)/1e6:.1f} MB/s)')
        inventory[source.name]=identity
    save_json(destination/'copy.json',inventory)
    return fingerprint(inventory)

def decode_init(blob, path, n):
    global _blob, _image_array
    _blob = open(blob,'rb')
    _image_array = np.memmap(path,mode='r+',dtype=np.uint8,shape=(n,224,224,3))

def decode_one(item):
    index, offset, nbytes, crc = item
    _blob.seek(offset); data=_blob.read(nbytes)
    if len(data)!=nbytes or (zlib.crc32(data)&0xffffffff)!=crc: raise IOError(f'JPEG CRC/size error at image {index}')
    with Image.open(io.BytesIO(data)) as image:
        if image.size!=(416,416): raise ValueError(f'Image {index} is {image.size}, expected 416x416')
        _image_array[index]=np.asarray(image.convert('RGB').resize((224,224),resample=Image.Resampling.BILINEAR))
    return index

def decode_images(pack, cache, key):
    table=pd.read_parquet(pack/'packed/images.parquet').sort_values('scene').reset_index(drop=True)
    if table['scene'].duplicated().any(): raise ValueError('Duplicate scenes in image index')
    n=len(table); path=cache/'img224.u8'; meta=cache/'images.json'
    decode_key=fingerprint([key,'PIL-RGB-BILINEAR-224-v1',sha_file(pack/'packed/images.parquet')])
    if path.is_file() and meta.is_file() and path.stat().st_size==n*224*224*3 and read_json(meta).get('key')==decode_key:
        save_json(cache/'scene_to_index.json',{scene:i for i,scene in enumerate(table['scene'])})
        log(f'image cache: reused N={n}'); return n,decode_key
    path.unlink(missing_ok=True); pending=path.with_name(path.name+'.partial'); pending.unlink(missing_ok=True)
    require_free(cache,n*224*224*3+500_000_000,'224 uint8 image cache')
    initial=np.memmap(pending,mode='w+',dtype=np.uint8,shape=(n,224,224,3)); initial.flush(); del initial
    jobs=[(i,int(row.offset),int(row.nbytes),int(row.crc32)) for i,row in enumerate(table.itertuples(index=False))]
    started=time.perf_counter();done=0
    with mp.get_context('spawn').Pool(4,initializer=decode_init,
            initargs=(str(pack/'packed/images.bin'),str(pending),n)) as pool:
        for _ in pool.imap_unordered(decode_one,jobs,chunksize=32):
            done+=1
            if done%1000==0 or done==n:
                log(f'decode images {done}/{n}; {done/max(time.perf_counter()-started,1e-6):.1f} images/s')
    flushed=np.memmap(pending,mode='r+',dtype=np.uint8,shape=(n,224,224,3));flushed.flush();del flushed
    pending.replace(path)
    save_json(cache/'scene_to_index.json',{scene:i for i,scene in enumerate(table['scene'])})
    save_json(meta,{'key':decode_key,'N':n,'shape':[n,224,224,3],'dtype':'uint8'})
    return n,decode_key

def fix_manifests(root,cache):
    from collections import Counter
    frames={};source_hashes={}
    for name in ('train','val','test_seen','test_unseen'):
        payload=(root/(name+'.csv')).read_bytes()
        import hashlib
        source_hashes[name]=hashlib.sha256(payload).hexdigest()
        frames[name]=pd.read_csv(io.BytesIO(payload),dtype={'stem':str,'scene':str,'obj':str,'part':str},keep_default_na=False)
        frames[name]['obj_name']=frames[name]['prompt'].map(normalize_name)
    unseen=set(frames['test_unseen']['obj_name']);removals={}
    for name in ('train','val'):
        frame=frames[name];bad=frame['obj_name'].isin(unseen)
        top=Counter(frame.loc[bad,'obj_name']).most_common(20)
        removals[name]={'before':len(frame),'removed':int(bad.sum()),'top20':top}
        frames[name]=frame.loc[~bad].reset_index(drop=True)
        log(f'leakage fix {name}: removed {int(bad.sum())}/{len(frame)}; remaining={len(frames[name])}; top20={top}')
        if frames[name].empty: raise RuntimeError(f'No {name} rows remain after the grip/name leakage fix')
    test_scenes=set(frames['test_seen']['scene'])|set(frames['test_unseen']['scene'])
    assert not set(frames['train']['scene'])&set(frames['val']['scene'])
    for name in ('train','val'):
        assert not set(frames[name]['scene'])&test_scenes,'Source manifests leak official test scenes'
        assert not set(frames[name]['obj_name'])&unseen
    for name,frame in frames.items():frame.to_csv(cache/(name+'.csv'),index=False)
    save_json(cache/'manifest_fix.json',{'source_sha256':source_hashes,'unseen_names':sorted(unseen),'removals':removals,
                                      'counts':{name:len(frame) for name,frame in frames.items()},'version':'grip-v1'})
    return frames,fingerprint([source_hashes,'grip-v1'])

def load_clip(device):
    from transformers import CLIPModel
    from transformers.utils import logging as transformers_logging
    transformers_logging.disable_progress_bar()
    model=CLIPModel.from_pretrained(CLIP_ID,torch_dtype=torch.float16,cache_dir='/tmp/ga_hf',
                                   token=os.environ.get('HF_TOKEN') or False).to(device)
    model.eval().requires_grad_(False)
    log(f'CLIP cache encoder: trainable=0; frozen={sum(p.numel() for p in model.parameters()):,}; device={device}; dtype=fp16')
    return model

def text_embeddings(frames,cache,key,batch_size=256):
    prompts=sorted(set().union(*(set(frame['prompt']) for frame in frames.values())))
    text_key=fingerprint([key,CLIP_ID,prompts,'projected-text-v1'])
    path,meta=cache/'text_emb.npy',cache/'text.json'
    if path.is_file() and meta.is_file() and read_json(meta).get('key')==text_key:
        array=np.load(path,mmap_mode='r');
        if array.shape==(len(prompts),512) and array.dtype==np.float16:
            save_json(cache/'prompt_to_row.json',{prompt:i for i,prompt in enumerate(prompts)})
            log(f'text cache reused P={len(prompts)}');return len(prompts)
    require_free(cache,len(prompts)*512*2+200_000_000,'CLIP text embeddings')
    from transformers import AutoTokenizer
    device=torch.device('cuda:0');model=load_clip(device)
    tokenizer=AutoTokenizer.from_pretrained(CLIP_ID,cache_dir='/tmp/ga_hf',token=os.environ.get('HF_TOKEN') or False)
    pending=path.with_name(path.name+'.partial')
    array=np.lib.format.open_memmap(pending,mode='w+',dtype=np.float16,shape=(len(prompts),512))
    with torch.inference_mode():
        for step,start in enumerate(range(0,len(prompts),batch_size),1):
            tokens=tokenizer(prompts[start:start+batch_size],padding=True,truncation=True,max_length=77,return_tensors='pt')
            output=model.get_text_features(**{k:v.to(device) for k,v in tokens.items()})
            # Transformers 4.x returns a Tensor; newer versions return projected pooler_output.
            features=output if isinstance(output,torch.Tensor) else output.pooler_output
            if features.shape!=(min(batch_size,len(prompts)-start),512):raise ValueError('Unexpected CLIP text feature shape')
            if not torch.isfinite(features).all():raise FloatingPointError('Nonfinite frozen CLIP text features')
            array[start:start+len(features)]=features.float().cpu().numpy().astype(np.float16)
            if step%25==0 or start+len(features)==len(prompts):log(f'CLIP text {start+len(features)}/{len(prompts)} prompts')
    array.flush();del array,model;gc.collect();torch.cuda.empty_cache()
    pending.replace(path)
    save_json(cache/'prompt_to_row.json',{prompt:i for i,prompt in enumerate(prompts)})
    save_json(meta,{'key':text_key,'P':len(prompts),'shape':[len(prompts),512],'dtype':'float16','model':CLIP_ID})
    return len(prompts)

def token_worker(gpu,start,stop,n,cache_string,batch_size):
    cache=Path(cache_string);torch.cuda.set_device(gpu);torch.set_num_threads(2)
    device=torch.device(f'cuda:{gpu}');model=load_clip(device)
    images=np.memmap(cache/'img224.u8',mode='r',dtype=np.uint8,shape=(n,224,224,3))
    tokens=np.memmap(cache/'clip_tokens.f16.partial',mode='r+',dtype=np.float16,shape=(n,196,768))
    mean=torch.tensor([.48145466,.4578275,.40821073],device=device)[None,:,None,None]
    std=torch.tensor([.26862954,.26130258,.27577711],device=device)[None,:,None,None]
    with torch.inference_mode():
        for step,begin in enumerate(range(start,stop,batch_size),1):
            end=min(begin+batch_size,stop)
            image=torch.from_numpy(np.array(images[begin:end],copy=True)).to(device).permute(0,3,1,2).float()/255
            pixel_values=((image-mean)/std).half()
            hidden=model.vision_model(pixel_values=pixel_values).last_hidden_state[:,1:,:]
            features=model.vision_model.post_layernorm(hidden)
            if features.shape!=(end-begin,196,768):raise ValueError('Expected CLIP (B,196,768) patch tokens')
            tokens[begin:end]=features.cpu().numpy().astype(np.float16,copy=False)
            if step%50==0 or end==stop:log(f'GPU {gpu} CLIP patches {end-start}/{stop-start}')
    tokens.flush();del tokens,images,model

def patch_tokens(cache,n,key,batch_size=64):
    path,meta=cache/'clip_tokens.f16',cache/'tokens.json'
    token_key=fingerprint([key,CLIP_ID,'post-layernorm-patches-v1'])
    size=n*196*768*2
    if path.is_file() and path.stat().st_size==size and meta.is_file() and read_json(meta).get('key')==token_key:
        log(f'CLIP patch cache reused: {size/1e9:.2f} GB');return
    if torch.cuda.device_count()<2:raise RuntimeError('C/D token preparation requires both GPUs (T4 x2)')
    path.unlink(missing_ok=True);pending=path.with_name(path.name+'.partial');pending.unlink(missing_ok=True)
    require_free(cache,25_000_000_000,'C/D patch tokens require at least 25 GB FREE on /tmp')
    initial=np.memmap(pending,mode='w+',dtype=np.float16,shape=(n,196,768));initial.flush();del initial
    context=mp.get_context('spawn');middle=(n+1)//2
    processes=[context.Process(target=token_worker,args=(gpu,start,end,n,str(cache),batch_size))
               for gpu,start,end in ((0,0,middle),(1,middle,n))]
    for process in processes:process.start()
    for process in processes:process.join()
    if any(process.exitcode!=0 for process in processes):raise RuntimeError('A CLIP token-cache GPU worker failed; cache is incomplete')
    pending.replace(path)
    save_json(meta,{'key':token_key,'N':n,'shape':[n,196,768],'dtype':'float16','bytes':size,'model':CLIP_ID})

def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--root',required=True);parser.add_argument('--variants',nargs='+',default=['A','B'])
    parser.add_argument('--pack-root',default='/tmp/ga_pack');parser.add_argument('--cache',default='/tmp/ga_cache')
    args=parser.parse_args()
    if not torch.cuda.is_available():raise RuntimeError('Preparation needs GPU access for frozen fp16 CLIP; select T4 x2')
    root,pack,cache=Path(args.root),Path(args.pack_root),Path(args.cache)
    cache.mkdir(parents=True,exist_ok=True);pack.mkdir(parents=True,exist_ok=True)
    started=time.perf_counter();copy_key=copy_pack(root,pack)
    n,image_key=decode_images(pack,cache,copy_key)
    frames,manifest_key=fix_manifests(root,cache)
    p=text_embeddings(frames,cache,manifest_key)
    if any(run_variant(spec) in ('C','D','E') for spec in args.variants):patch_tokens(cache,n,image_key)
    else:log('A/B only: no CLIP patch-token cache allocated')
    save_json(cache/'prepared.json',{'N':n,'P':p,'variants':args.variants,'image_key':image_key,
        'manifest_key':manifest_key,'metric_sha256':sha_file(Path(__file__).parent/'eval_utils.py'),
        'seconds':time.perf_counter()-started})
    log(f'PREPARE complete: N={n}, P={p}, elapsed={time.perf_counter()-started:.1f}s')

if __name__=='__main__':main()


## Cell 2.4 — Write `code/data.py`


In [ ]:
%%writefile /kaggle/working/code/data.py
"""All per-sample reads use local memory maps; no geometric/image augmentation."""
import json
import os
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from eval_utils import load_gt_array, rect_polygon
from common import read_json, worker_seed

def draw_targets(gts_416, cfg):
    targets = np.zeros((4, cfg.img, cfg.img), dtype=np.float32)
    scale = cfg.img/416.0
    for gt in gts_416:
        cx, cy, length, width, angle = map(float, gt)
        cx, cy, length, width = cx*scale, cy*scale, length*scale, width*scale
        polygon = rect_polygon(cx, cy, max(length/3, cfg.min_rect_px), max(width, cfg.min_rect_px), angle)
        corners = np.rint(np.asarray(polygon.exterior.coords)[:4]).astype(np.int32)
        mask = np.zeros((cfg.img, cfg.img), dtype=np.uint8)
        cv2.fillPoly(mask, [corners], 1)
        inside = mask.astype(bool)
        targets[0, inside] = 1.0
        targets[1, inside] = np.cos(2*angle)
        targets[2, inside] = np.sin(2*angle)
        targets[3, inside] = np.clip(length/cfg.width_norm, 0, 1)
    return targets

class GraspDataset(Dataset):
    def __init__(self, manifest_csv, split, cfg):
        self.cfg, self.split = cfg, split
        self.frame = manifest_csv.reset_index(drop=True).copy() if isinstance(manifest_csv, pd.DataFrame) else pd.read_csv(
            manifest_csv, dtype={'stem': str, 'scene': str, 'obj': str, 'part': str}, keep_default_na=False)
        self.rows = self.frame.to_dict('records')
        self.scene_to_idx = read_json(Path(cfg.cache)/'scene_to_index.json')
        self.prompt_to_idx = read_json(Path(cfg.cache)/'prompt_to_row.json')
        self.n_images = int(read_json(Path(cfg.cache)/'images.json')['N'])
        self._pid = None

    def _open(self):
        if self._pid != os.getpid():
            self._labels = np.load(Path(self.cfg.pack_root)/'packed/labels.npy', mmap_mode='r', allow_pickle=False)
            self._images = np.memmap(Path(self.cfg.cache)/'img224.u8', mode='r', dtype=np.uint8,
                                     shape=(self.n_images, self.cfg.img, self.cfg.img, 3))
            self._tokens = None
            if self.cfg.variant in ('C', 'D', 'E'):
                self._tokens = np.memmap(Path(self.cfg.cache)/'clip_tokens.f16', mode='r', dtype=np.float16,
                                        shape=(self.n_images, 196, 768))
            self._pid = os.getpid()

    def __len__(self): return len(self.rows)

    def __getitem__(self, index):
        self._open()
        row = self.rows[index]
        scene_idx = int(self.scene_to_idx[row['scene']])
        start, count = int(row['label_start']), int(row['label_count'])
        gts = load_gt_array(self._labels[start:start+count]).astype(np.float32)
        result = {'img': torch.from_numpy(np.array(self._images[scene_idx].transpose(2, 0, 1), copy=True)),
                  'prompt_idx': int(self.prompt_to_idx[row['prompt']]), 'scene_idx': scene_idx,
                  'targets': torch.from_numpy(draw_targets(gts, self.cfg)), 'gts_416': gts,
                  'stem': row['stem'], 'scene': row['scene'], 'prompt': row['prompt']}
        if self._tokens is not None:
            result['clip_tokens'] = torch.from_numpy(np.array(self._tokens[scene_idx], copy=True))
        return result

def collate(samples):
    result = {key: torch.stack([item[key] for item in samples]) for key in ('img', 'targets')}
    for key in ('prompt_idx', 'scene_idx'): result[key] = torch.tensor([item[key] for item in samples], dtype=torch.long)
    for key in ('gts_416', 'stem', 'scene', 'prompt'): result[key] = [item[key] for item in samples]
    if 'clip_tokens' in samples[0]: result['clip_tokens'] = torch.stack([item['clip_tokens'] for item in samples])
    return result

def make_loader(manifest, split, cfg, shuffle=False, batch_size=None):
    dataset = GraspDataset(manifest, split, cfg)
    generator = torch.Generator().manual_seed(cfg.seed)
    return DataLoader(dataset, batch_size=batch_size or cfg.batch, shuffle=shuffle, num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(), collate_fn=collate, worker_init_fn=worker_seed,
        persistent_workers=cfg.num_workers > 0, generator=generator, drop_last=False,
        **({'multiprocessing_context':'spawn'} if cfg.num_workers > 0 else {}))


## Cell 2.5 — Write `code/model.py`


In [ ]:
%%writefile /kaggle/working/code/model.py
import math
import torch
from torch import nn
import torch.nn.functional as F

def conv(cin, cout, kernel, stride=1, padding=0, transpose=False):
    layer = nn.ConvTranspose2d if transpose else nn.Conv2d
    return nn.Sequential(layer(cin, cout, kernel, stride, padding), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class ResidualBlock(nn.Module):
    def __init__(self, channels=128):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1), nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True), nn.Conv2d(channels, channels, 3, padding=1), nn.BatchNorm2d(channels))
    def forward(self, features): return features+self.layers(features)

class FiLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.affine = nn.Linear(512, 256)
        nn.init.zeros_(self.affine.weight); nn.init.zeros_(self.affine.bias)
    def forward(self, features, text):
        with torch.autocast(device_type=features.device.type, enabled=False):
            dgamma, beta = self.affine(text.float()).chunk(2, dim=-1)
        return features*(1+dgamma.to(features.dtype)[:, :, None, None])+beta.to(features.dtype)[:, :, None, None]

class LanguageGate(nn.Module):
    """S = cos(Wp P, Wt t). With text_free=True (variant E, 'C0'), the query Wt t is replaced by one learned,
    prompt-independent vector q, so the gate still sees CLIP patch features but cannot depend on the instruction."""
    def __init__(self, text_free=False):
        super().__init__()
        self.text_free = text_free
        self.Wp = nn.Linear(768, 256)
        if text_free: self.query = nn.Parameter(torch.randn(256)/16.0)
        else: self.Wt = nn.Linear(512, 256)
        self.log_tau = nn.Parameter(torch.tensor(math.log(10.0)))
    def forward(self, features, text, tokens):
        if tokens is None or tokens.shape[1:] != (196, 768):
            raise ValueError('C/D require cached patch tokens (B,196,768)')
        with torch.autocast(device_type=features.device.type, enabled=False):
            patches = F.normalize(self.Wp(tokens.float().reshape(-1, 14, 14, 768)), dim=-1)
            if self.text_free: language = F.normalize(self.query.float(), dim=-1)[None, :].expand(patches.shape[0], -1)
            else: language = F.normalize(self.Wt(text.float()), dim=-1)
            similarity = (patches*language[:, None, None, :]).sum(-1)
            logits = self.log_tau.exp()*similarity[:, None, :, :]
            attention = F.interpolate(logits.sigmoid(), size=(56, 56), mode='bilinear', align_corners=False)
        return features*(1+attention.to(features.dtype)), logits

class GraspNet(nn.Module):
    def __init__(self, variant='A'):
        super().__init__()
        if variant not in ('A','B','C','D','E'): raise ValueError(variant)
        self.variant = variant
        self.conv1, self.conv2, self.conv3 = conv(3,32,9,padding=4), conv(32,64,4,2,1), conv(64,128,4,2,1)
        self.residuals = nn.ModuleList([ResidualBlock() for _ in range(5)])
        self.up1, self.up2, self.final = conv(128,64,4,2,1,True), conv(64,32,4,2,1,True), conv(32,32,9,padding=4)
        self.dropout = nn.Dropout(0.1)
        self.heads = nn.ModuleDict({key:nn.Conv2d(32,1,1) for key in ('q','cos','sin','width')})
        # Shared backbone initializes before any variant-specific layers.
        self.films = nn.ModuleList([FiLM() for _ in range(5)]) if variant != 'A' else nn.ModuleList()
        self.gate = LanguageGate(text_free=(variant=='E')) if variant in ('C','D','E') else None
        self.register_buffer('mean', torch.tensor([.485,.456,.406])[None,:,None,None])
        self.register_buffer('std', torch.tensor([.229,.224,.225])[None,:,None,None])

    def forward(self, image, text=None, tokens=None, return_shapes=False):
        shapes = {}
        x = (image.float()/255.0-self.mean)/self.std
        for name in ('conv1','conv2','conv3'):
            x = getattr(self,name)(x)
            if return_shapes: shapes[name] = tuple(x.shape)
        t = F.normalize(text.float(), dim=-1) if self.variant != 'A' else None
        logits = None
        if self.gate is not None: x, logits = self.gate(x,t,tokens)
        if return_shapes: shapes['pre_residual'] = tuple(x.shape)
        for i, block in enumerate(self.residuals):
            x = block(x)
            if return_shapes: shapes[f'residual_{i}'] = tuple(x.shape)
            if self.films:
                x = self.films[i](x,t)
                if return_shapes: shapes[f'film_{i}'] = tuple(x.shape)
        for name in ('up1','up2','final'):
            x = getattr(self,name)(x)
            if return_shapes: shapes[name] = tuple(x.shape)
        x = self.dropout(x)
        if return_shapes: shapes['dropout'] = tuple(x.shape)
        results = []
        for name,head in self.heads.items():
            result = head(x); results.append(result)
            if return_shapes: shapes['head_'+name] = tuple(result.shape)
        maps = torch.cat(results, dim=1)
        if return_shapes: shapes['maps'] = tuple(maps.shape)
        return {'maps':maps, 'gate_logits':logits, 'shapes':shapes}

def loss_components(output, targets, cfg):
    losses = {name:F.smooth_l1_loss(output['maps'][:,i],targets[:,i])
              for i,name in enumerate(('q','cos','sin','width'))}
    gate = output['maps'].new_zeros(())
    if cfg.variant == 'D':
        target14 = (F.adaptive_max_pool2d(targets[:,0:1], (14,14)) > 0).float()
        gate = F.binary_cross_entropy_with_logits(output['gate_logits'], target14)
    losses['gate'] = gate
    losses['total'] = sum(losses[name] for name in ('q','cos','sin','width'))+cfg.lambda_gate*gate
    return losses

def parameter_counts(model):
    return {'trainable':sum(p.numel() for p in model.parameters() if p.requires_grad),
            'frozen':sum(p.numel() for p in model.parameters() if not p.requires_grad)}


## Cell 2.6 — Write `code/infer.py`


In [ ]:
%%writefile /kaggle/working/code/infer.py
import torch
from torchvision.transforms.functional import gaussian_blur

def post_process(maps):
    # Reshape B*C into a single leading dimension for torchvision version compatibility.
    b, c, h, w = maps.shape
    qcs = gaussian_blur(maps[:,:3].float().reshape(b*3,h,w), [9,9], [2.,2.]).reshape(b,3,h,w)
    width = gaussian_blur(maps[:,3].float(), [5,5], [1.,1.])[:,None]
    return torch.cat([qcs,width],dim=1)

def decode_top1(maps, cfg):
    smooth = post_process(maps)
    b, _, h, w = smooth.shape
    indices = smooth[:,0].reshape(b,-1).argmax(dim=1)
    r, c = indices//w, indices%w
    batch = torch.arange(b,device=maps.device)
    cosine, sine = smooth[batch,1,r,c], smooth[batch,2,r,c]
    angle = .5*torch.atan2(sine,cosine)
    length = (smooth[batch,3,r,c]*cfg.width_norm).clamp(min=1.0)
    scale = cfg.img/416.0
    return torch.stack([c.float()/scale,r.float()/scale,length/scale,length/(2*scale),angle],dim=1)


## Cell 2.7 — Write `code/evaluate.py`


In [ ]:
%%writefile /kaggle/working/code/evaluate.py
"""Metric calls are imported from the dataset's unchanged eval_utils.py."""
import csv
import dataclasses
import warnings
warnings.filterwarnings('ignore',category=FutureWarning)
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from common import log, model_inputs
from data import make_loader
from infer import decode_top1
from eval_utils import is_success, rect_polygon

def evaluate(model, manifest, cfg, limit=None, *, out_csv=None, device=None):
    frame=manifest.copy() if isinstance(manifest,pd.DataFrame) else pd.read_csv(manifest,dtype={'scene':str,'stem':str},keep_default_na=False)
    if limit is not None and len(frame)>limit:frame=frame.sample(n=limit,random_state=0)
    frame=frame.reset_index(drop=True)
    if frame.empty:raise ValueError('Cannot evaluate an empty manifest')
    device=device or next(model.parameters()).device
    name=Path(manifest).stem if not isinstance(manifest,pd.DataFrame) else 'validation'
    output=Path(out_csv or Path(cfg.out)/(name+'_predictions.csv'));output.parent.mkdir(parents=True,exist_ok=True)
    loader=make_loader(frame,name,dataclasses.replace(cfg,num_workers=0))  # in-process: no worker spawn per evaluation
    text=np.load(Path(cfg.cache)/'text_emb.npy',mmap_mode='r',allow_pickle=False)
    was_training=model.training;model.eval();successes=total=0
    with output.open('w',newline='',encoding='utf-8') as stream,torch.inference_mode():
        writer=csv.writer(stream);writer.writerow(['stem','prompt','success','pred_cx','pred_cy','pred_length','pred_width','pred_angle','n_gt'])
        for step,batch in enumerate(loader,1):
            inputs=model_inputs(batch,text,cfg.variant,device)
            with torch.cuda.amp.autocast(enabled=cfg.amp and device.type=='cuda',dtype=torch.float16):
                maps=model(*inputs)['maps']
            predictions=decode_top1(maps,cfg).cpu().numpy()
            for stem,prompt,pred,gts in zip(batch['stem'],batch['prompt'],predictions,batch['gts_416']):
                good=bool(is_success(pred,gts));successes+=int(good);total+=1
                writer.writerow([stem,prompt,int(good),*map(float,pred),len(gts)])
            if step%50==0 or total==len(frame):log(f'eval {name}: {total}/{len(frame)} success={successes/total:.4f}')
    model.train(was_training)
    return successes/total

def latency_ms(model,manifest,cfg,warmup=20,iterations=100):
    """Batch-1 cached-input forward + GPU blur/decode, excluding data loading and metric."""
    device=next(model.parameters()).device
    if device.type!='cuda':raise RuntimeError('Reported inference latency requires CUDA synchronization')
    batch=next(iter(make_loader(manifest,'latency',dataclasses.replace(cfg,num_workers=0),batch_size=1)))
    text=np.load(Path(cfg.cache)/'text_emb.npy',mmap_mode='r')
    inputs=model_inputs(batch,text,cfg.variant,device)
    model.eval();times=[]
    with torch.inference_mode():
        for i in range(warmup+iterations):
            torch.cuda.synchronize(device);started=time.perf_counter()
            with torch.cuda.amp.autocast(enabled=cfg.amp,dtype=torch.float16):maps=model(*inputs)['maps']
            decode_top1(maps,cfg)
            torch.cuda.synchronize(device)
            if i>=warmup:times.append((time.perf_counter()-started)*1000)
    return float(np.mean(times))

def qualitative(model,cfg,path):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.collections import LineCollection
    import textwrap
    import packed_io
    frames=[]
    for split in ('test_seen','test_unseen'):
        frame=pd.read_csv(Path(cfg.cache)/(split+'.csv'),dtype={'scene':str,'stem':str},keep_default_na=False)
        chosen=frame.sample(n=4,replace=len(frame)<4,random_state=0).copy();chosen['display_split']=split;frames.append(chosen)
    samples=pd.concat(frames,ignore_index=True)
    loader=make_loader(samples,'qualitative',dataclasses.replace(cfg,num_workers=0),batch_size=8)
    batch=next(iter(loader));device=next(model.parameters()).device
    text=np.load(Path(cfg.cache)/'text_emb.npy',mmap_mode='r');model.eval()
    with torch.inference_mode(),torch.cuda.amp.autocast(enabled=cfg.amp and device.type=='cuda',dtype=torch.float16):
        predictions=decode_top1(model(*model_inputs(batch,text,cfg.variant,device))['maps'],cfg).cpu().numpy()
    figure,axes=plt.subplots(2,4,figsize=(16,9),layout='constrained')
    for index,(ax,pred,gts) in enumerate(zip(axes.flat,predictions,batch['gts_416'])):
        with packed_io.read_jpeg(cfg.pack_root,batch['scene'][index]) as image:ax.imshow(np.asarray(image))
        ax.add_collection(LineCollection([np.asarray(rect_polygon(*gt).exterior.coords) for gt in gts],colors='lime',linewidths=.6,alpha=.65))
        valid=np.isfinite(pred).all() and np.all(pred[2:4]>0)
        if valid:
            points=np.asarray(rect_polygon(*pred).exterior.coords);ax.plot(points[:,0],points[:,1],color='red',linewidth=2)
        status='success' if is_success(pred,gts) else 'failure'
        title=f'{samples.iloc[index]["display_split"]} | {status}\n'+textwrap.fill(batch['prompt'][index],42)
        if not valid:title+='\ninvalid predicted dimensions'
        ax.set_title(title,fontsize=9);ax.set_xlim(-.5,415.5);ax.set_ylim(415.5,-.5);ax.axis('off')
    figure.savefig(path,dpi=130);plt.close(figure)
    log(f'qualitative figure saved: {path}')


## Cell 2.8 — Write `code/train.py`


In [ ]:
%%writefile /kaggle/working/code/train.py
import argparse
import warnings
warnings.filterwarnings('ignore',category=FutureWarning)
import csv
import math
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from config import Config
from common import log,read_json,save_json,seed_all,model_inputs,fingerprint,sha_file
from data import make_loader
from model import GraspNet,loss_components,parameter_counts
from evaluate import evaluate,latency_ms,qualitative
from eval_utils import harmonic_mean

def lr_multiplier(step,total,warmup):
    if warmup and step<warmup:return (step+1)/warmup
    progress=min(1.,max(0.,(step-warmup)/max(1,total-warmup)))
    return .5*(1+math.cos(math.pi*progress))

def finalize(model,cfg,out,device,counts,identity,best,best_epoch,step,stop_reason,started,metric_sha,val_path):
    checkpoint=torch.load(out/'best.pt',map_location=device,weights_only=True)
    model.load_state_dict(checkpoint['model']);model.eval()
    limit=200 if cfg.smoke else None
    val=evaluate(model,val_path,cfg,limit,out_csv=out/'val_predictions.csv',device=device)
    seen=evaluate(model,Path(cfg.cache)/'test_seen.csv',cfg,limit,out_csv=out/'test_seen_predictions.csv',device=device)
    unseen=evaluate(model,Path(cfg.cache)/'test_unseen.csv',cfg,limit,out_csv=out/'test_unseen_predictions.csv',device=device)
    latency=latency_ms(model,Path(cfg.cache)/'test_seen.csv',cfg)
    qualitative(model,cfg,out/'qualitative.png')
    eval_counts={name:min(200,len(pd.read_csv(Path(cfg.cache)/(name+'.csv')))) if cfg.smoke
                 else len(pd.read_csv(Path(cfg.cache)/(name+'.csv'))) for name in ('val','test_seen','test_unseen')}
    metrics={'run_id':identity,'variant':cfg.variant,'smoke':cfg.smoke,'params':sum(counts.values()),**counts,
        'val':val,'seen':seen,'unseen':unseen,'H':harmonic_mean(seen,unseen),'latency_ms':latency,
        'latency_scope':'batch1 cached-input forward + GPU blur/decode; warmup20, timed100, per-iteration CUDA sync',
        'best_epoch':best_epoch,'best_selection_val':best,'steps':step,'wall_seconds':time.perf_counter()-started,
        'stop_reason':stop_reason,'eval_counts':eval_counts,'metric_sha256':metric_sha,'config':cfg.to_dict()}
    save_json(out/'metrics.json',metrics);log(f'{cfg.variant} DONE: val={val:.4f} seen={seen:.4f} unseen={unseen:.4f} '
                                               f'H={metrics["H"]:.4f} latency={latency:.2f}ms best_epoch={best_epoch}')

def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--variant',choices=list('ABCDE'),required=True);parser.add_argument('--seed',type=int,default=0)
    parser.add_argument('--gpu',type=int,required=True);parser.add_argument('--out',required=True)
    parser.add_argument('--smoke',action='store_true');parser.add_argument('--epochs',type=int,default=6)
    parser.add_argument('--max-hours',type=float,default=3.5);parser.add_argument('--batch',type=int,default=32)
    parser.add_argument('--cache',default='/tmp/ga_cache');parser.add_argument('--pack-root',default='/tmp/ga_pack')
    parser.add_argument('--num-workers',type=int,default=2);parser.add_argument('--force',action='store_true')
    parser.add_argument('--eval-only',action='store_true',help='skip training; evaluate out/best.pt (used by the watchdog)')
    args=parser.parse_args()
    cfg=Config(variant=args.variant,seed=args.seed,epochs=args.epochs,max_hours=args.max_hours,batch=args.batch,smoke=args.smoke,
               out=args.out,cache=args.cache,pack_root=args.pack_root,num_workers=args.num_workers).validate()
    if not torch.cuda.is_available() or args.gpu>=torch.cuda.device_count():raise RuntimeError('Requested CUDA GPU is unavailable')
    torch.cuda.set_device(args.gpu);device=torch.device(f'cuda:{args.gpu}');torch.set_num_threads(2);seed_all(cfg.seed)
    torch.backends.cudnn.benchmark=True;torch.backends.cudnn.deterministic=False  # speed; unit tests stay deterministic
    out=Path(cfg.out);out.mkdir(parents=True,exist_ok=True)
    prepared=read_json(Path(cfg.cache)/'prepared.json')
    roundtrip=read_json(Path(cfg.cache)/'target_roundtrip.json')
    if roundtrip.get('rate',0)<.80 or roundtrip.get('manifest_key')!=prepared['manifest_key']:
        raise RuntimeError('Run tests.py successfully on the current prepared validation cache before training')
    metric_sha=sha_file(Path(__file__).parent/'eval_utils.py')
    if roundtrip['metric_sha256']!=metric_sha:raise RuntimeError('Metric changed after target round-trip validation')
    source_sha={path.name:sha_file(path) for path in Path(__file__).parent.glob('*.py')}
    identity=fingerprint([{k:v for k,v in cfg.to_dict().items() if k!='out'},
                          prepared['image_key'],prepared['manifest_key'],source_sha])
    if (out/'metrics.json').is_file() and not args.force:
        if read_json(out/'metrics.json')['run_id']!=identity:raise RuntimeError('Output contains a different run; choose --out or use --force')
        log(f'{cfg.variant}: completed matching run already exists; reusing results');return
    if args.eval_only:
        if not (out/'best.pt').is_file():raise FileNotFoundError(out/'best.pt')
        started=time.perf_counter();model=GraspNet(cfg.variant).to(device);counts=parameter_counts(model)
        ck=torch.load(out/'best.pt',map_location=device,weights_only=True)
        log(f'{cfg.variant}: EVAL-ONLY on best.pt (epoch {ck.get("best_epoch")}, step {ck.get("step")})')
        finalize(model,cfg,out,device,counts,identity,float(ck.get('val_success',float('nan'))),int(ck.get('best_epoch',0)),
                 int(ck.get('step',0)),'watchdog_eval_only',started,metric_sha,Path(cfg.cache)/'val.csv')
        return
    started=time.perf_counter();model=GraspNet(cfg.variant).to(device)
    counts=parameter_counts(model)
    log(f'variant={cfg.variant}; GPU={args.gpu} {torch.cuda.get_device_name(args.gpu)}; trainable={counts["trainable"]:,}; '
        f'frozen={counts["frozen"]:,}; CLIP remains external/frozen; smoke={cfg.smoke}')
    save_json(out/'config.json',cfg.to_dict())
    train_path=Path(cfg.cache)/'train.csv';val_path=Path(cfg.cache)/'val.csv'
    loader=make_loader(train_path,'train',cfg,shuffle=True)
    if not len(loader):raise RuntimeError('Empty training loader')
    text=np.load(Path(cfg.cache)/'text_emb.npy',mmap_mode='r')
    optimizer=torch.optim.AdamW(model.parameters(),lr=cfg.lr,weight_decay=cfg.wd)
    max_steps=300 if cfg.smoke else cfg.epochs*len(loader)
    epochs=max(cfg.epochs,math.ceil(max_steps/len(loader))) if cfg.smoke else cfg.epochs
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda step:lr_multiplier(step,max_steps,cfg.warmup_steps))
    scaler=torch.cuda.amp.GradScaler(enabled=cfg.amp)
    val_frame=pd.read_csv(val_path,dtype={'scene':str,'stem':str},keep_default_na=False)
    fixed_n=min(200 if cfg.smoke else cfg.val_subset,len(val_frame))
    fixed_val=val_frame.sample(n=fixed_n,random_state=0).reset_index(drop=True)  # same selection subset for every seed
    save_json(out/'fixed_val_stems.json',fixed_val['stem'].tolist())
    best=-1.;best_epoch=0;step=0;stop_reason='epochs';window_images=0;window_start=time.perf_counter()
    fields=['step','epoch','event','loss','q','cos','sin','width','gate','lr','val_success','imgs_s','elapsed_s']
    with (out/'train_log.csv').open('w',newline='',encoding='utf-8') as stream:
        writer=csv.DictWriter(stream,fieldnames=fields);writer.writeheader()
        for epoch in range(1,epochs+1):
            model.train();loss_values={};clock_stop=False
            for batch in loader:
                optimizer.zero_grad(set_to_none=True)
                targets=batch['targets'].to(device,non_blocking=True)
                with torch.cuda.amp.autocast(enabled=cfg.amp,dtype=torch.float16):
                    output=model(*model_inputs(batch,text,cfg.variant,device));losses=loss_components(output,targets,cfg)
                if not torch.isfinite(losses['total']):raise FloatingPointError('Nonfinite training loss')
                old_scale=scaler.get_scale();scaler.scale(losses['total']).backward();scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scaler.step(optimizer);scaler.update()
                if scaler.get_scale()>=old_scale:scheduler.step()
                step+=1;window_images+=len(batch['img'])
                loss_values={name:float(value.detach().float().item()) for name,value in losses.items()}
                elapsed=time.perf_counter()-started;rate=window_images/max(time.perf_counter()-window_start,1e-6)
                writer.writerow({'step':step,'epoch':epoch,'event':'train','loss':loss_values['total'],
                    **{k:loss_values[k] for k in ('q','cos','sin','width','gate')},'lr':optimizer.param_groups[0]['lr'],
                    'val_success':'','imgs_s':rate,'elapsed_s':elapsed})
                if step%cfg.log_every==0:
                    log(f'{cfg.variant} step={step} epoch={epoch} losses={loss_values} lr={optimizer.param_groups[0]["lr"]:.6g} imgs/s={rate:.1f}')
                    stream.flush();window_images=0;window_start=time.perf_counter()
                clock_stop=elapsed>cfg.max_hours*3600
                if step>=max_steps or clock_stop:break
            # Complete this evaluation even if the wall-time budget elapsed mid-epoch/evaluation.
            val_success=evaluate(model,fixed_val,cfg,out_csv=out/f'val_epoch_{epoch:02d}.csv',device=device)
            writer.writerow({'step':step,'epoch':epoch,'event':'val','lr':optimizer.param_groups[0]['lr'],
                             'val_success':val_success,'elapsed_s':time.perf_counter()-started})
            stream.flush();log(f'{cfg.variant} epoch={epoch} step={step} fixed_val_success={val_success:.4f}')
            if val_success>best:
                best,best_epoch=val_success,epoch
                pending=out/'best.pt.partial'
                torch.save({'model':model.state_dict(),'config':cfg.to_dict(),'best_epoch':epoch,
                            'val_success':best,'step':step,'metric_sha256':metric_sha},pending)
                pending.replace(out/'best.pt')
            if clock_stop or time.perf_counter()-started>cfg.max_hours*3600:stop_reason='wall_time';break
            if step>=max_steps:stop_reason='smoke_steps' if cfg.smoke else 'epochs';break
            window_images=0;window_start=time.perf_counter()
    finalize(model,cfg,out,device,counts,identity,best,best_epoch,step,stop_reason,started,metric_sha,val_path)

if __name__=='__main__':main()


## Cell 2.9 — Write `code/tests.py`


In [ ]:
%%writefile /kaggle/working/code/tests.py
"""Run before training: exact architecture/FiLM checks and a real-data target round trip."""
import argparse
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from common import log,read_json,save_json,sha_file,seed_all
from config import Config
from model import GraspNet,FiLM,loss_components,parameter_counts
from data import make_loader
from infer import decode_top1
from eval_utils import is_success

def test_shapes_and_film(device):
    torch.set_num_threads(2)
    image=torch.randint(0,256,(2,3,224,224),dtype=torch.uint8,device=device)
    text=torch.randn(2,512,device=device);tokens=torch.randn(2,196,768,device=device)
    expected={'conv1':(2,32,224,224),'conv2':(2,64,112,112),'conv3':(2,128,56,56),
        'pre_residual':(2,128,56,56),'up1':(2,64,112,112),'up2':(2,32,224,224),
        'final':(2,32,224,224),'maps':(2,4,224,224)}
    for i in range(5):expected[f'residual_{i}']=(2,128,56,56)
    expected['dropout']=(2,32,224,224)
    for name in ('q','cos','sin','width'):expected['head_'+name]=(2,1,224,224)
    base_output=None
    for variant in 'ABCDE':
        seed_all(0);model=GraspNet(variant).to(device).eval()
        with torch.inference_mode():output=model(image,text,tokens,return_shapes=True)
        for name,shape in expected.items():assert output['shapes'][name]==shape,(variant,name,output['shapes'][name])
        if variant!='A':
            for i in range(5):assert output['shapes'][f'film_{i}']==(2,128,56,56)
        if variant in 'CDE':
            assert output['gate_logits'].shape==(2,1,14,14)
            assert abs(model.gate.log_tau.exp().item()-10)<1e-4
        if variant=='A':base_output=output['maps'].clone()
        if variant=='B':assert torch.equal(base_output,output['maps']),'B must equal A at zero-FiLM initialization'
        log(f'PASS shapes {variant}: {parameter_counts(model)}')
    layer=FiLM().to(device);features=torch.randn(2,128,56,56,device=device)
    assert torch.equal(layer(features,text),features)
    log('PASS FiLM identity at initialization')

def target_roundtrip(cfg,device):
    frame=pd.read_csv(Path(cfg.cache)/'val.csv',dtype={'scene':str,'stem':str},keep_default_na=False)
    if len(frame)<200:raise RuntimeError(f'Target round-trip requires 200 validation rows, found {len(frame)}')
    chosen=frame.sample(n=200,random_state=0).reset_index(drop=True)
    loader=make_loader(chosen,'target_roundtrip',cfg)
    records=[]
    with torch.inference_mode():
        for step,batch in enumerate(loader,1):
            predictions=decode_top1(batch['targets'].to(device),cfg).cpu().numpy()
            for stem,pred,gts in zip(batch['stem'],predictions,batch['gts_416']):
                records.append({'stem':stem,'success':int(is_success(pred,gts)),'n_gt':len(gts),
                    **{name:float(value) for name,value in zip(('cx','cy','length','width','angle'),pred)}})
            if step%10==0 or len(records)==200:log(f'target round-trip {len(records)}/200')
    rate=sum(row['success'] for row in records)/200
    pd.DataFrame(records).to_csv(Path(cfg.cache)/'target_roundtrip.csv',index=False)
    prepared=read_json(Path(cfg.cache)/'prepared.json')
    save_json(Path(cfg.cache)/'target_roundtrip.json',{'rate':rate,'n':200,'threshold':.80,'seed':0,
        'manifest_key':prepared['manifest_key'],'metric_sha256':sha_file(Path(__file__).parent/'eval_utils.py')})
    log(f'target round-trip (oracle decode upper bound): {rate:.3%}; required >=80%; diagnostics: {cfg.cache}/target_roundtrip.csv')
    assert rate>=.80,'Target round-trip failed: inspect target_roundtrip.csv before training; metric/target rules were not relaxed'
    return rate

def main():
    parser=argparse.ArgumentParser();parser.add_argument('--device',default='cuda:0')
    parser.add_argument('--cache',default='/tmp/ga_cache');parser.add_argument('--pack-root',default='/tmp/ga_pack')
    args=parser.parse_args();device=torch.device(args.device)
    cfg=Config(cache=args.cache,pack_root=args.pack_root,num_workers=0)
    test_shapes_and_film(device);target_roundtrip(cfg,device);log('ALL UNIT CHECKS PASSED')

if __name__=='__main__':main()


## Cell 2.10 — Write `code/launch.py`


In [ ]:
%%writefile /kaggle/working/code/launch.py
"""Run variants in two-GPU waves with a hard watchdog so the notebook always finishes and saves outputs."""
import argparse
import datetime
import json
import os
import subprocess
import sys
import time
from pathlib import Path
from common import log,run_variant,run_seed

def last_lines(path,n=4):
    with Path(path).open('rb') as stream:
        stream.seek(0,2);size=stream.tell();stream.seek(max(0,size-16384))
        lines=stream.read().decode('utf-8',errors='replace').splitlines()
    return [line for line in lines if 'Warning' not in line and 'autocast' not in line and 'GradScaler' not in line][-n:]

def spawn(variant,gpu,out,args,extra=()):
    stream=(out/'stdout.log').open('a',encoding='utf-8',buffering=1)
    stream.write(f'\nLAUNCH {datetime.datetime.now().isoformat()} variant={variant} GPU={gpu} extra={list(extra)}\n')
    command=[sys.executable,'-u',str(Path(__file__).parent/'train.py'),'--variant',run_variant(variant),
             '--seed',str(run_seed(variant)),'--gpu',str(gpu),
             '--out',str(out),'--cache',args.cache,'--pack-root',args.pack_root,'--max-hours',str(args.max_hours),*extra]
    if args.smoke:command.append('--smoke')
    env=os.environ.copy();env.update(PYTHONUNBUFFERED='1',OMP_NUM_THREADS='2',MKL_NUM_THREADS='2',PYTHONWARNINGS='ignore::FutureWarning')
    try:
        process=subprocess.Popen(command,stdout=stream,stderr=subprocess.STDOUT,env=env,cwd=Path(__file__).parent)
    except BaseException:
        stream.close();raise
    return process,stream

def stop(process):
    if process.poll() is None:
        process.terminate()
        try:process.wait(timeout=30)
        except subprocess.TimeoutExpired:process.kill();process.wait()

def run_wave(variants,args,base):
    status={};children=[]
    hard=(args.max_hours+args.grace_hours)*3600;started=time.monotonic()
    try:
        for gpu,variant in enumerate(variants):
            out=base/variant;out.mkdir(parents=True,exist_ok=True)
            process,stream=spawn(variant,gpu,out,args);children.append([variant,gpu,process,stream,out])
            log(f'Launched {variant} on GPU {gpu}; hard watchdog at {hard/3600:.2f} h')
        next_tail=time.monotonic()+args.interval;timed_out=False
        while any(child[2].poll() is None for child in children):
            if time.monotonic()-started>hard:
                timed_out=True;log('WATCHDOG: wall-time limit reached; terminating training processes')
                for child in children:stop(child[2])
                break
            if time.monotonic()>=next_tail:
                for variant,gpu,process,stream,out in children:
                    stream.flush();log(f'--- {variant} GPU{gpu} status={process.poll()} elapsed={(time.monotonic()-started)/3600:.2f}h ---')
                    for line in last_lines(out/'stdout.log'):log(f'[{variant}] {line}')
                next_tail=time.monotonic()+args.interval
            time.sleep(5)
        for variant,gpu,process,stream,out in children:
            stream.flush();code=process.returncode
            status[variant]={'returncode':code,'watchdog':timed_out and code!=0,'metrics':(out/'metrics.json').is_file()}
            log(f'[{variant}] exit={code}; metrics.json={status[variant]["metrics"]}')
            for line in last_lines(out/'stdout.log',8):log(f'[{variant}] {line}')
        # Final evaluation for any variant that has a checkpoint but no metrics (watchdog kill or late crash).
        evals=[]
        for variant,gpu,process,stream,out in children:
            if not (out/'metrics.json').is_file() and (out/'best.pt').is_file():
                log(f'[{variant}] running eval-only on best.pt (GPU {gpu})')
                proc,s=spawn(variant,gpu,out,args,('--eval-only',));evals.append((variant,proc,s,out))
        deadline=time.monotonic()+args.eval_minutes*60
        for variant,proc,s,out in evals:
            try:proc.wait(timeout=max(1,deadline-time.monotonic()))
            except subprocess.TimeoutExpired:log(f'[{variant}] eval-only timed out');stop(proc)
            s.close();status[variant]['eval_only_returncode']=proc.returncode;status[variant]['metrics']=(out/'metrics.json').is_file()
            for line in last_lines(out/'stdout.log',6):log(f'[{variant}] {line}')
    finally:
        for variant,gpu,process,stream,out in children:
            stop(process)
            if not stream.closed:stream.close()
    return status

def main():
    parser=argparse.ArgumentParser();parser.add_argument('--variants',nargs='+',default=['A','B'])
    parser.add_argument('--smoke',action='store_true');parser.add_argument('--base',default='/kaggle/working/runs')
    parser.add_argument('--cache',default='/tmp/ga_cache');parser.add_argument('--pack-root',default='/tmp/ga_pack')
    parser.add_argument('--max-hours',type=float,default=3.5);parser.add_argument('--grace-hours',type=float,default=0.5)
    parser.add_argument('--eval-minutes',type=float,default=30);parser.add_argument('--interval',type=float,default=600)
    args=parser.parse_args()
    if len(set(args.variants))!=len(args.variants):raise ValueError('Variants must be unique')
    for spec in args.variants:run_variant(spec)
    import torch
    if torch.cuda.device_count()<2:raise RuntimeError('Select Kaggle GPU T4 x2 before launching parallel runs')
    base=Path(args.base);base.mkdir(parents=True,exist_ok=True);status={}
    for begin in range(0,len(args.variants),2):status.update(run_wave(args.variants[begin:begin+2],args,base))
    (base/'launch_status.json').write_text(json.dumps(status,indent=2))
    missing=[v for v,s in status.items() if not s['metrics']]
    log(f'Launch finished. status={status}' + (f' | WITHOUT metrics: {missing}' if missing else ''))

if __name__=='__main__':main()


## Cell 2.11 — Write `code/summarize.py`


In [ ]:
%%writefile /kaggle/working/code/summarize.py
import argparse
import shutil
from pathlib import Path
import pandas as pd
from common import log,read_json,working_budget

def summarize(variants,base='/kaggle/working/runs',destination='/kaggle/working/results',cache='/tmp/ga_cache'):
    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True);rows=[]
    for variant in variants:
        source=Path(base)/variant;out=destination/variant;out.mkdir(parents=True,exist_ok=True)
        if (source/'metrics.json').is_file():
            metric=read_json(source/'metrics.json')
            row={key:metric.get(key) for key in ('variant','params','trainable','frozen','val','seen','unseen','H','latency_ms','best_epoch','smoke','stop_reason')}
            row['run']=variant;row['seed']=metric.get('config',{}).get('seed')
            row['wall_hours']=metric['wall_seconds']/3600;row['status']='complete'
        else:
            row={'variant':variant,'status':'NO metrics.json'}
            log_path=source/'train_log.csv'
            if log_path.is_file():
                frame=pd.read_csv(log_path);vals=frame.loc[frame['event']=='val']
                if len(vals):row.update(best_fixed_val=float(vals['val_success'].max()),epochs_done=int(vals['epoch'].max()))
        rows.append(row)
        for name in ['metrics.json','train_log.csv','qualitative.png','best.pt','config.json','fixed_val_stems.json','stdout.log',
                     'val_predictions.csv','test_seen_predictions.csv','test_unseen_predictions.csv']:
            path=source/name
            if path.is_file():working_budget(path.stat().st_size);shutil.copyfile(path,out/name)
        log(f'Collected {variant}: {row}')
    frame=pd.DataFrame(rows);frame.to_csv(destination/'summary.csv',index=False)
    for name in ('prepared.json','manifest_fix.json','target_roundtrip.json','target_roundtrip.csv'):
        path=Path(cache)/name
        if path.is_file():working_budget(path.stat().st_size);shutil.copyfile(path,destination/name)
    if (Path(base)/'launch_status.json').is_file():shutil.copyfile(Path(base)/'launch_status.json',destination/'launch_status.json')
    working_budget();log(frame.to_string(index=False));return frame

def main():
    parser=argparse.ArgumentParser();parser.add_argument('--variants',nargs='+',default=['A','B'])
    parser.add_argument('--base',default='/kaggle/working/runs');parser.add_argument('--destination',default='/kaggle/working/results')
    parser.add_argument('--cache',default='/tmp/ga_cache')
    args=parser.parse_args();summarize(args.variants,args.base,args.destination,args.cache)

if __name__=='__main__':main()


### latency_e2e.py — batch-1 end-to-end latency (incl. frozen CLIP encoders) for A–E in one process


In [ ]:
%%writefile /kaggle/working/code/latency_e2e.py
"""End-to-end batch-1 latency for every variant, measured sequentially in ONE process on ONE idle GPU.

Stages (CUDA-synchronised, warmup 20, timed 100, fp16):
  head   = GraspNet forward + GPU blur/decode on cached inputs (what metrics.json reports as latency_ms)
  text   = CLIP tokenizer (CPU) + CLIP text encoder        (variants B-E; once per instruction)
  vision = CLIP ViT-B/16 image encoder -> 196 patch tokens  (variants C/D/E; once per image)
  e2e    = text + vision + head in one loop (uint8 224x224 image and raw prompt string -> grasp)
GraspNet weights are randomly initialised: run time does not depend on weight values.
"""
import argparse
import json
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from common import log, read_json, seed_all
from config import Config
from model import GraspNet, parameter_counts
from infer import decode_top1
from prepare import load_clip, CLIP_ID

def timed(fn, device, warmup=20, iterations=100):
    times = []
    for i in range(warmup + iterations):
        torch.cuda.synchronize(device); start = time.perf_counter()
        fn()
        torch.cuda.synchronize(device)
        if i >= warmup: times.append((time.perf_counter() - start) * 1000)
    return float(np.mean(times)), float(np.std(times))

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--cache', default='/tmp/ga_cache'); parser.add_argument('--gpu', type=int, default=0)
    parser.add_argument('--variants', nargs='+', default=list('ABCDE'))
    parser.add_argument('--out', default='/kaggle/working/results')
    args = parser.parse_args()
    device = torch.device(f'cuda:{args.gpu}'); torch.cuda.set_device(device); seed_all(0)
    torch.backends.cudnn.benchmark = True
    cache = Path(args.cache); cfg = Config()
    from transformers import AutoTokenizer
    import os
    tokenizer = AutoTokenizer.from_pretrained(CLIP_ID, cache_dir='/tmp/ga_hf', token=os.environ.get('HF_TOKEN') or False)
    clip = load_clip(device)
    n = int(read_json(cache / 'images.json')['N'])
    images = np.memmap(cache / 'img224.u8', mode='r', dtype=np.uint8, shape=(n, 224, 224, 3))
    row = pd.read_csv(cache / 'test_seen.csv', dtype={'scene': str, 'stem': str}, keep_default_na=False).iloc[0]
    scene_idx = int(read_json(cache / 'scene_to_index.json')[row['scene']]); prompt = str(row['prompt'])
    image_hwc = torch.from_numpy(np.array(images[scene_idx], copy=True)).to(device)          # uint8 (224,224,3)
    image = image_hwc.permute(2, 0, 1)[None].contiguous()                                   # uint8 (1,3,224,224)
    mean = torch.tensor([.48145466, .4578275, .40821073], device=device)[None, :, None, None]
    std = torch.tensor([.26862954, .26130258, .27577711], device=device)[None, :, None, None]

    def encode_text():
        tokens = tokenizer([prompt], padding=True, truncation=True, max_length=77, return_tensors='pt')
        output = clip.get_text_features(**{k: v.to(device) for k, v in tokens.items()})
        features = output if isinstance(output, torch.Tensor) else output.pooler_output
        return features.float()

    def encode_image():
        pixel_values = ((image.float() / 255 - mean) / std).half()
        hidden = clip.vision_model(pixel_values=pixel_values).last_hidden_state[:, 1:, :]
        return clip.vision_model.post_layernorm(hidden)

    rows = []
    with torch.inference_mode():
        text_cached = encode_text(); tokens_cached = encode_image()
        clip_text_params = sum(p.numel() for p in clip.text_model.parameters()) + sum(p.numel() for p in clip.text_projection.parameters())
        clip_vision_params = sum(p.numel() for p in clip.vision_model.parameters())
        for variant in args.variants:
            model = GraspNet(variant).to(device).eval()
            uses_text, uses_vision = variant != 'A', variant in ('C', 'D', 'E')
            t_in = text_cached if uses_text else None; v_in = tokens_cached if uses_vision else None

            def head():
                with torch.autocast('cuda', dtype=torch.float16):
                    maps = model(image, t_in, v_in)['maps']
                decode_top1(maps, cfg)

            def e2e():
                t = encode_text() if uses_text else None
                v = encode_image() if uses_vision else None
                with torch.autocast('cuda', dtype=torch.float16):
                    maps = model(image, t, v)['maps']
                decode_top1(maps, cfg)

            head_ms, head_sd = timed(head, device)
            text_ms, _ = timed(encode_text, device) if uses_text else (0.0, 0.0)
            vision_ms, _ = timed(encode_image, device) if uses_vision else (0.0, 0.0)
            e2e_ms, e2e_sd = timed(e2e, device)
            counts = parameter_counts(model)
            frozen = (clip_text_params if uses_text else 0) + (clip_vision_params if uses_vision else 0)
            rows.append({'variant': variant, 'trainable_params': counts['trainable'], 'frozen_clip_params_used': frozen,
                         'head_ms': head_ms, 'head_sd': head_sd, 'clip_text_ms': text_ms, 'clip_vision_ms': vision_ms,
                         'e2e_ms': e2e_ms, 'e2e_sd': e2e_sd})
            log(f'LATENCY {variant}: head={head_ms:.2f}+-{head_sd:.2f} text={text_ms:.2f} vision={vision_ms:.2f} '
                f'e2e={e2e_ms:.2f}+-{e2e_sd:.2f} ms; trainable={counts["trainable"]:,} frozen_used={frozen:,}')
            del model; torch.cuda.empty_cache()
    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)
    frame = pd.DataFrame(rows); frame.to_csv(out / 'latency_e2e.csv', index=False)
    (out / 'latency_e2e.json').write_text(json.dumps({'gpu': torch.cuda.get_device_name(device), 'batch': 1, 'dtype': 'fp16',
        'warmup': 20, 'iterations': 100, 'prompt': prompt, 'note': __doc__, 'rows': rows}, indent=2))
    log('\n' + frame.to_string(index=False))

if __name__ == '__main__':
    main()


## Cell 2.12 — Write `code/requirements.txt`


In [ ]:
%%writefile /kaggle/working/code/requirements.txt
torch>=2.2
torchvision>=0.17
transformers>=4.37
numpy
pandas
pyarrow
Pillow
shapely
opencv-python-headless
matplotlib


## Cell 2.13 — Write `code/.gitignore`


In [ ]:
%%writefile /kaggle/working/code/.gitignore
__pycache__/
.pytest_cache/
*.pt
*.partial
*.u8
*.f16
*.npy
results/
data/
.env


## Cell 2.14 — Write `code/README.md`


In [ ]:
%%writefile /kaggle/working/code/README.md
# Language-driven grasp detection ablations

The Kaggle notebook generates this directory as the reusable source repository.
`eval_utils.py` and `packed_io.py` are copied verbatim from the attached packed dataset.
The supplied evaluator is the sole success-metric implementation.

Use two CUDA GPUs. Preparation uses `/tmp/ga_pack`, `/tmp/ga_cache`, and `/tmp/ga_hf`.
Training uses `/tmp/ga_runs/<variant>`. Only source and collected results go under
`/kaggle/working`, subject to a 15 GB limit. C/D patch-token construction checks for
25 GB **free** on `/tmp` before allocating its approximately 14 GB token map.

```bash
python prepare.py --root /path/to/ga_eval --variants A B
python tests.py --device cuda:0
python launch.py --variants A B
python summarize.py --variants A B
```

For one run: `python train.py --variant B --gpu 0 --out /tmp/ga_runs/B`.
Add `--smoke` to the launcher/trainer for 300 training steps, 200 validation samples
and 200 examples per test split. Preparation still builds the full image/text caches.
Choose C/D in `--variants` to prepare patch tokens on both GPUs. Longer variant lists
run in waves of two. No CLIP parameters are included in or trained with the grasp model.

A ignores language; B adds identity-initialized FiLM after each residual block;
C adds the unsupervised language gate; D also supervises gate logits with compact-target
occupancy. The original length/width/angle target and decode rules are used without augmentation.

The fixed 200-validation-example target round-trip must reach 95% using the supplied
metric before training. Diagnostics are saved to `/tmp/ga_cache/target_roundtrip.csv`.
Failures can reveal geometry errors or target/decode representation limits; samples
and thresholds are not changed to force the test to pass.

Best checkpoints are selected on the same seed-0 validation subset each epoch.
Wall-time stopping completes the pending validation, then final evaluations always run.
Reported latency includes batch-1 forward and GPU smoothing/decode on cached inputs,
after 20 warmup iterations with per-iteration CUDA synchronization, and excludes
loading, GT metric computation and cache preparation.

All own progress uses flushed plain print statements; the launcher tails logs every 60s.
Matching completed runs and valid preprocessing caches are reused; `--force` overwrites
a trainer output. Keep smoke results separate from full results (the flag is saved).


## Cell 3 — Prepare once and run unit checks


In [ ]:
# CELL 3 — Configuration, one-time preparation, then the mandatory unit checks.
VARIANTS=['E','C_s1']  # E = C0 (text-free CLIP gate control); C_s1 = C with seed 1
SMOKE=False
import re
if not VARIANTS or len(set(VARIANTS))!=len(VARIANTS) or any(re.fullmatch(r'[ABCDE](_s\d+)?',v) is None for v in VARIANTS):
    raise ValueError('VARIANTS must be unique run specs: A-E, optionally with _s<seed> (e.g. C_s1).')
subprocess.run([sys.executable,'-u',str(CODE/'prepare.py'),'--root',str(ROOT),'--variants',*VARIANTS],
               cwd=CODE,check=True)
subprocess.run([sys.executable,'-u',str(CODE/'tests.py'),'--device','cuda:0'],cwd=CODE,check=True)
print('Preparation and architecture/FiLM/200-validation-target round-trip checks passed.',flush=True)


## Cell 4 — Launch the two-GPU experiments


In [ ]:
# CELL 4 — First two variants in parallel on GPU 0 and GPU 1; additional variants use later waves.
command=[sys.executable,'-u',str(CODE/'launch.py'),'--variants',*VARIANTS,'--max-hours','3.5','--grace-hours','0.5']
if SMOKE:command.append('--smoke')
subprocess.run(command,cwd=CODE,check=False)  # never block summarize/outputs
print('All requested training subprocesses completed; failure in either process would raise.',flush=True)
# Both GPUs are idle now: measure batch-1 end-to-end latency for A-E sequentially on GPU 0 (never blocks outputs).
if not SMOKE:
    subprocess.run([sys.executable,'-u',str(CODE/'latency_e2e.py'),'--gpu','0'],cwd=CODE,check=False)


## Cell 5 — Collect results and summary


In [ ]:
# CELL 5 — Collect portable outputs and display the cross-variant summary.
subprocess.run([sys.executable,'-u',str(CODE/'summarize.py'),'--variants',*VARIANTS],cwd=CODE,check=True)
import pandas as pd
summary=pd.read_csv('/kaggle/working/results/summary.csv')
display(summary)
used=sum(p.stat().st_size for p in Path('/kaggle/working').rglob('*') if p.is_file())
assert used<15_000_000_000,f'Working outputs exceeded 15 GB: {used:,} bytes'
print(f'Outputs: /kaggle/working/results/summary.csv and results/<variant>/; working bytes={used:,}',flush=True)
print('Source repo: /kaggle/working/code/. No training/cache data need to be committed.',flush=True)
